# Tech Challenge Fase 3 — Etapa 1: Dataset

## Assistente Virtual de Saúde da Mulher

**O que esta etapa faz:**
1. Instala as bibliotecas necessárias
2. Baixa o dataset MedQuAD (Q&A médico real do NIH)
3. Baixa o dataset Menstrual Health (HuggingFace)
4. Carrega o dataset sintético criado para este projeto
5. Combina, valida e salva o dataset final

> Execute as células em ordem, de cima para baixo.

## Celula 1 — Instalacao das bibliotecas

In [1]:
# ============================================================
# TECH CHALLENGE FASE 3 — Setup inicial
# ============================================================
!pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    langgraph \
    chromadb \
    sentence-transformers \
    python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

## Celula 2 — Configuracao da chave OpenRouter

Crie sua chave gratuita em: https://openrouter.ai/workspaces/default/keys

No Google Colab: clique no icone de chave no menu lateral esquerdo,
adicione um segredo chamado OPENROUTER_API_KEY com sua chave.

In [2]:
from google.colab import userdata
import os

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
print("Chave configurada ✓")

Chave configurada ✓


## Célula 3 — Primeiro teste: Model do OpenRouter respondendo

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openrouter/free",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0.2,
)

from langchain_core.messages import HumanMessage
resposta = llm.invoke([HumanMessage(content="Quais são os sintomas clássicos de uma sepse?")])
print(resposta.content)


A **sepse** (ou sepse) é uma complicação grave que pode surgir após uma infecção, quando a resposta do corpo à infecção prejudica seus próprios tecidos e órgãos. Se não tratada rapidamente, pode evoluir para **shock séptico** (insuficiência circulatória) e até à morte. Os **sintomas clássicos da sepse** incluem:

---

### **Sintomas gerais:**
1. **Febre alta ou hipotermia**  
   - Febre acima de 38,5°C (101,3°F) ou, em alguns casos (especialmente em idosos ou imunodeprimidos), temperatura abaixo de 36°C (96,8°F).  
   - Calafrios intensos, mesmo sem suor.

2. **Taquicardia (coração acelerado)**  
   - Frequência cardíaca acima de 90 batimentos por minuto.  
   - O coração bombeia mais rápido para compensar a circulação prejudicada.

3. **Taquipneia (respiração acelerada)**  
   - Respiração rápida (mais de 24 respirações por minuto) ou dispneia.  
   - Pode haver hipoxia (baixa concentração de oxigênio no sangue).

4. **Alterações no estado mental**  
   - Confusão, agitação, dificuld

## Célula 4 — Instalar dependência extra

In [7]:
# Célula 4 — Instalar biblioteca HuggingFace datasets
!pip install -q datasets

## Célula 5 — Carregar dataset sintético (pt-BR)

In [8]:
# Célula 5 — Carregar dataset sintético em português
import json
from google.colab import drive

drive.mount('/content/drive')

with open("/content/drive/MyDrive/TechChallengeFase3_IA_Devs/dataset_saude_mulher.json", "r", encoding="utf-8") as f:
    dataset_sintetico = json.load(f)

# Adiciona idioma pt-BR (ausente no JSON original)
for item in dataset_sintetico:
    item["idioma"] = "pt-BR"

print(f"Dataset sintético carregado: {len(dataset_sintetico)} registros")
print(f"Categorias: {set(d['categoria'] for d in dataset_sintetico)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset sintético carregado: 35 registros
Categorias: {'prevencao_geral', 'saude_ginecologica', 'contracepcao', 'violencia_domestica', 'saude_reprodutiva', 'saude_mental', 'infeccoes_sexualmente_transmissiveis', 'prevencao_cancer', 'amamentacao', 'menopausa', 'gestacao'}


## Célula 6 — Carregar e padronizar MedQuAD + Menstrual Health (inglês)

In [9]:
# Célula 6 — Carregar datasets externos do HuggingFace
from datasets import load_dataset
import pandas as pd

# ── MedQuAD ──────────────────────────────────────────────
print("Baixando MedQuAD... (pode demorar 1-2 minutos)")
medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
df_medquad = pd.DataFrame(medquad)
print(f"MedQuAD carregado: {len(df_medquad)} registros | Colunas: {list(df_medquad.columns)}")

# Filtra por palavras-chave de saúde da mulher
palavras_chave = [
    "women", "female", "gynecolog", "obstetric", "pregnancy",
    "menstrual", "menopause", "breast", "ovarian", "uterine",
    "cervical", "vaginal", "prenatal", "postpartum", "maternal",
    "contracepti", "fertility", "endometriosis", "ovary",
    "mammogram", "hormone", "estrogen", "progesterone"
]

col_p = "Question" if "Question" in df_medquad.columns else "question"
col_r = "Answer"   if "Answer"   in df_medquad.columns else "answer"

df_filtrado = df_medquad[
    df_medquad[col_p].apply(
        lambda t: isinstance(t, str) and any(p in t.lower() for p in palavras_chave)
    )
].copy()

print(f"Registros filtrados (saúde da mulher): {len(df_filtrado)}")

medquad_padronizado = []
for i, row in df_filtrado.iterrows():
    p = str(row[col_p]).strip()
    r = str(row[col_r]).strip()
    if not p or not r or p == "nan" or r == "nan":
        continue
    medquad_padronizado.append({
        "id":               f"MEDQUAD_{i:05d}",
        "categoria":        "saude_geral_feminina",
        "pergunta":         p,
        "resposta":         r,
        "fonte":            "MedQuAD - NIH",
        "nivel_confianca":  0.90,
        "encaminhamento":   "medico",
        "urgencia":         "baixa",
        "idioma":           "en"
    })

print(f"MedQuAD padronizado: {len(medquad_padronizado)} registros")

# ── Menstrual Health ──────────────────────────────────────
print("\nBaixando Menstrual Health Awareness Dataset...")
menstrual_padronizado = []

try:
    ds = load_dataset("gjyotk/Menstrual-Health-Awareness-Dataset", split="train")
    df_m = pd.DataFrame(ds)
    print(f"Colunas: {list(df_m.columns)}")

    possiveis_p = ["question", "Question", "input", "prompt", "text"]
    possiveis_r = ["answer",   "Answer",   "output", "response"]
    cols = list(df_m.columns)
    cp = next((c for c in possiveis_p if c in df_m.columns), cols[0])
    cr = next((c for c in possiveis_r if c in df_m.columns), cols[1] if len(cols) > 1 else cols[0])

    for i, row in df_m.iterrows():
        p = str(row[cp]).strip()
        r = str(row[cr]).strip()
        if p and r and p != "nan" and r != "nan":
            menstrual_padronizado.append({
                "id":               f"MENSTRUAL_{i:04d}",
                "categoria":        "saude_menstrual",
                "pergunta":         p,
                "resposta":         r,
                "fonte":            "Menstrual Health Awareness Dataset",
                "nivel_confianca":  0.88,
                "encaminhamento":   "ginecologista",
                "urgencia":         "baixa",
                "idioma":           "en"
            })
    print(f"Menstrual Health padronizado: {len(menstrual_padronizado)} registros")

except Exception as e:
    print(f"Aviso: não foi possível carregar o dataset Menstrual Health — {e}")
    print("Continuando sem este dataset.")

Baixando MedQuAD... (pode demorar 1-2 minutos)
MedQuAD carregado: 16407 registros | Colunas: ['qtype', 'Question', 'Answer']
Registros filtrados (saúde da mulher): 281
MedQuAD padronizado: 281 registros

Baixando Menstrual Health Awareness Dataset...


train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/530 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/45 [00:00<?, ? examples/s]

Colunas: ['instruction (string)', 'output (string)']
Menstrual Health padronizado: 530 registros


## Célula 7 — Combinar, validar e limpar

In [10]:
# Célula 7 — Combinar todos os datasets e validar qualidade

def validar_e_limpar(dataset):
    limpo = []
    removidos = 0
    for item in dataset:
        p = str(item.get("pergunta", "")).strip()
        r = str(item.get("resposta", "")).strip()
        if not p or not r or p == "nan" or r == "nan":
            removidos += 1
            continue
        if len(p) < 10 or len(r) < 20:
            removidos += 1
            continue
        # Garante campos obrigatórios
        item.setdefault("nivel_confianca", 0.85)
        item.setdefault("encaminhamento",  "medico")
        item.setdefault("urgencia",        "baixa")
        item.setdefault("idioma",          "en")
        item.setdefault("categoria",       "saude_geral_feminina")
        item["pergunta"] = p
        item["resposta"] = r
        limpo.append(item)
    return limpo, removidos

# Combina na ordem: sintético → menstrual → MedQuAD (limitado a 500)
dataset_bruto = (
    dataset_sintetico
    + menstrual_padronizado
    + medquad_padronizado[:500]
)

dataset_validado, removidos = validar_e_limpar(dataset_bruto)

print(f"Sintético (pt-BR):      {len(dataset_sintetico)} registros")
print(f"Menstrual Health (en):  {len(menstrual_padronizado)} registros")
print(f"MedQuAD limitado (en):  {min(len(medquad_padronizado), 500)} registros")
print(f"─────────────────────────────────")
print(f"Total bruto:            {len(dataset_bruto)} registros")
print(f"Removidos na validação: {removidos}")
print(f"Total válido final:     {len(dataset_validado)} registros")

Sintético (pt-BR):      35 registros
Menstrual Health (en):  530 registros
MedQuAD limitado (en):  281 registros
─────────────────────────────────
Total bruto:            846 registros
Removidos na validação: 14
Total válido final:     832 registros


## Célula 8 — Salvar datasets e relatório final

In [11]:
# Célula 8 — Salvar datasets processados e exibir relatório final
import os

os.makedirs("/content/data/processed", exist_ok=True)  # cria a pasta se não existir

# Dataset completo
caminho_final = "/content/data/processed/dataset_final.json"
with open(caminho_final, "w", encoding="utf-8") as f:
    json.dump(dataset_validado, f, ensure_ascii=False, indent=2)

# Dataset somente pt-BR (sintético)
dataset_ptbr = [x for x in dataset_validado if x.get("idioma") == "pt-BR"]
caminho_ptbr = "/content/data/processed/dataset_ptbr.json"
with open(caminho_ptbr, "w", encoding="utf-8") as f:
    json.dump(dataset_ptbr, f, ensure_ascii=False, indent=2)

# Relatório
df_final = pd.DataFrame(dataset_validado)
print("=" * 50)
print("         RELATÓRIO FINAL DO DATASET")
print("=" * 50)
print(f"Total de registros:          {len(df_final)}")
print(f"\nPor idioma:")
print(df_final["idioma"].value_counts().to_string())
print(f"\nPor categoria:")
print(df_final["categoria"].value_counts().to_string())
print(f"\nPor urgência:")
print(df_final["urgencia"].value_counts().to_string())
print(f"\nNível de confiança médio:    {df_final['nivel_confianca'].mean():.3f}")
print(f"\nArquivos salvos:")
print(f"  → {caminho_final}  ({len(dataset_validado)} registros)")
print(f"  → {caminho_ptbr}  ({len(dataset_ptbr)} registros pt-BR)")
print(f"\n✓ Etapa 1 concluída! Próximo passo: Etapa 2 — RAG com LangChain + ChromaDB")

         RELATÓRIO FINAL DO DATASET
Total de registros:          832

Por idioma:
idioma
en       797
pt-BR     35

Por categoria:
categoria
saude_menstrual                         516
saude_geral_feminina                    281
saude_ginecologica                        6
gestacao                                  6
violencia_domestica                       5
contracepcao                              4
saude_mental                              3
prevencao_cancer                          3
menopausa                                 2
saude_reprodutiva                         2
amamentacao                               2
infeccoes_sexualmente_transmissiveis      1
prevencao_geral                           1

Por urgência:
urgencia
baixa    814
media     10
alta       8

Nível de confiança médio:    0.890

Arquivos salvos:
  → /content/data/processed/dataset_final.json  (832 registros)
  → /content/data/processed/dataset_ptbr.json  (35 registros pt-BR)

✓ Etapa 1 concluída! Próximo passo: E

## Etapa 2 — RAG com LangChain + ChromaDB

##Célula 9 — Indexar dataset no ChromaDB

In [12]:
# Célula 9 — Indexar dataset no ChromaDB (vector store)
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"  # modelo leve, roda no Colab sem GPU
)

cliente = chromadb.Client()
colecao = cliente.create_collection(
    name="saude_mulher",
    embedding_function=embedding_fn
)

# Carrega o dataset final salvo na etapa anterior
with open("/content/data/processed/dataset_final.json", "r", encoding="utf-8") as f:
    dataset_validado = json.load(f)

colecao.add(
    ids=[d["id"] for d in dataset_validado],
    documents=[
        f"Pergunta: {d['pergunta']}\nResposta: {d['resposta']}"
        for d in dataset_validado
    ],
    metadatas=[{
        "categoria":        d["categoria"],
        "fonte":            d["fonte"],
        "encaminhamento":   d["encaminhamento"],
        "urgencia":         d["urgencia"],
        "idioma":           d.get("idioma", "en"),
        "nivel_confianca":  str(d["nivel_confianca"])
    } for d in dataset_validado]
)

print(f"✓ {colecao.count()} documentos indexados no ChromaDB")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ 832 documentos indexados no ChromaDB


## Célula 10 — Testar busca vetorial

In [13]:
# Célula 10 — Validar busca por similaridade semântica
pergunta_teste = "paciente com ciclo menstrual irregular e acne"

resultados = colecao.query(
    query_texts=[pergunta_teste],
    n_results=3
)

print(f"Pergunta: {pergunta_teste}\n")
print("Top 3 documentos mais relevantes:\n")
for i, (doc, meta) in enumerate(zip(
    resultados["documents"][0],
    resultados["metadatas"][0]
)):
    print(f"[{i+1}] Categoria: {meta['categoria']} | Urgência: {meta['urgencia']} | Idioma: {meta['idioma']}")
    print(f"     Fonte: {meta['fonte']}")
    print(f"     Trecho: {doc[:150]}...")
    print()

Pergunta: paciente com ciclo menstrual irregular e acne

Top 3 documentos mais relevantes:

[1] Categoria: saude_menstrual | Urgência: baixa | Idioma: en
     Fonte: Menstrual Health Awareness Dataset
     Trecho: Pergunta: Does having irregular periods mean something is wrong with my reproductive system?
Resposta: Not necessarily. Irregular periods can be cause...

[2] Categoria: saude_menstrual | Urgência: baixa | Idioma: en
     Fonte: Menstrual Health Awareness Dataset
     Trecho: Pergunta: How does menopause affect women's health?
Resposta: Menopause, the natural cessation of menstruation, can cause symptoms such as hot flashes...

[3] Categoria: saude_geral_feminina | Urgência: baixa | Idioma: en
     Fonte: MedQuAD - NIH
     Trecho: Pergunta: What is (are) Polycystic ovarian syndrome ?
Resposta: Polycystic ovarian syndrome (PCOS) is a health problem that can affect a woman's menst...



## Célula 11 — Montar o RAG chain (busca + LLM)

In [14]:
# Célula 11 — RAG chain: recupera contexto do ChromaDB e envia ao LLM
from langchain_core.prompts import ChatPromptTemplate

def buscar_contexto(pergunta: str, n_resultados: int = 3) -> str:
    """Busca documentos relevantes no ChromaDB e retorna como texto."""
    resultados = colecao.query(
        query_texts=[pergunta],
        n_results=n_resultados
    )
    trechos = []
    for doc, meta in zip(resultados["documents"][0], resultados["metadatas"][0]):
        trechos.append(
            f"[Fonte: {meta['fonte']} | Categoria: {meta['categoria']}]\n{doc}"
        )
    return "\n\n---\n\n".join(trechos)


prompt_template = ChatPromptTemplate.from_messages([
    ("system", """Você é um assistente médico especializado em saúde da mulher.
Responda SEMPRE em português brasileiro, de forma clara e empática.
Use o contexto fornecido para embasar sua resposta.
Ao final, indique o encaminhamento recomendado quando relevante.
NUNCA invente informações — se não souber, diga que o paciente deve consultar um médico."""),
    ("human", """Contexto médico relevante:
{contexto}

Pergunta do profissional de saúde:
{pergunta}""")
])

def responder(pergunta: str) -> str:
    contexto = buscar_contexto(pergunta)
    chain = prompt_template | llm
    resposta = chain.invoke({
        "contexto": contexto,
        "pergunta": pergunta
    })
    return resposta.content


# Teste
pergunta = "Paciente de 28 anos com ciclos irregulares, acne e dificuldade para engravidar. O que investigar?"
print(f"Pergunta: {pergunta}\n")
print("Resposta do assistente:")
print("-" * 50)
print(responder(pergunta))

Pergunta: Paciente de 28 anos com ciclos irregulares, acne e dificuldade para engravidar. O que investigar?

Resposta do assistente:
--------------------------------------------------

Com base nos sintomas apresentados (ciclos irregulares, acne e dificuldade para engravidar), a investigação inicial deve focar nas principais causas de infertilidade associadas a esses achados, especialmente a **Síndrome dos Ovários Policísticos (SOP)**, que é uma das condições mais comuns nessa faixa etária.

### **Investigação Sugerida:**

1. **Avaliação hormonal basal (fase folicular inicial do ciclo, ou a qualquer momento se ciclos muito irregulares):**
   - Hormônio Folículo Estimulante (FSH)
   - Hormônio Luteinizante (LH)
   - Testosterona total
   - Prolactina
   - Hormônio Antimülleriano (AMH) – para avaliar reserva ovariana
   - Tireoide (TSH e T4 livre) – para excluir disfunção tireoidiana

2. **Exame físico e ultrassonografia transvaginal:**
   - Avaliação do aspecto ovariano (presença de múl

## Célula 12 — Definir o estado e os nós do grafo

In [15]:
# Célula 12 — LangGraph: definição do estado e nós
from langgraph.graph import StateGraph, END
from typing import TypedDict

# ── Estado compartilhado entre os nós ────────────────────
class EstadoConsulta(TypedDict):
    pergunta:       str
    urgencia:       str   # baixa | media | alta
    contexto:       str
    resposta:       str
    encaminhamento: str

# ── Nó 1: Triagem — classifica urgência da pergunta ──────
def no_triagem(estado: EstadoConsulta) -> EstadoConsulta:
    pergunta = estado["pergunta"].lower()

    palavras_alta = [
        "hemorragia", "desmaio", "convulsão", "sepse", "choque",
        "dor intensa", "sangramento intenso", "emergência", "inconsciente"
    ]
    palavras_media = [
        "febre", "dor", "corrimento", "irregularidade", "infecção",
        "suspeita", "investigar", "diagnosticar"
    ]

    if any(p in pergunta for p in palavras_alta):
        urgencia = "alta"
    elif any(p in pergunta for p in palavras_media):
        urgencia = "media"
    else:
        urgencia = "baixa"

    print(f"[Triagem] Urgência classificada: {urgencia}")
    return {**estado, "urgencia": urgencia}

# ── Nó 2: Busca RAG — recupera contexto do ChromaDB ──────
def no_busca_rag(estado: EstadoConsulta) -> EstadoConsulta:
    contexto = buscar_contexto(estado["pergunta"], n_resultados=3)
    print(f"[RAG] Contexto recuperado: {len(contexto)} caracteres")
    return {**estado, "contexto": contexto}

# ── Nó 3: Gerar resposta — chama o LLM ───────────────────
def no_gerar_resposta(estado: EstadoConsulta) -> EstadoConsulta:
    alerta = ""
    if estado["urgencia"] == "alta":
        alerta = "\n⚠️ ATENÇÃO: Caso de ALTA URGÊNCIA — considere atendimento imediato.\n"

    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""Você é um assistente médico especializado em saúde da mulher.
Responda SEMPRE em português brasileiro, de forma clara e empática.
Use o contexto fornecido para embasar sua resposta.
Ao final, indique o encaminhamento recomendado.
NUNCA invente informações — se não souber, oriente consulta médica.{alerta}"""),
        ("human", """Contexto médico relevante:
{contexto}

Pergunta:
{pergunta}""")
    ])

    chain = prompt | llm
    resposta = chain.invoke({
        "contexto": estado["contexto"],
        "pergunta": estado["pergunta"]
    })

    print(f"[LLM] Resposta gerada: {len(resposta.content)} caracteres")
    return {**estado, "resposta": resposta.content}

# ── Nó 4: Formatar saída ─────────────────────────────────
def no_formatar_saida(estado: EstadoConsulta) -> EstadoConsulta:
    urgencia_emoji = {"alta": "🔴", "media": "🟡", "baixa": "🟢"}
    emoji = urgencia_emoji.get(estado["urgencia"], "⚪")

    print("\n" + "=" * 60)
    print(f"  ASSISTENTE DE SAÚDE DA MULHER  {emoji} Urgência: {estado['urgencia'].upper()}")
    print("=" * 60)
    print(estado["resposta"])
    print("=" * 60)

    return estado

print("✓ Nós definidos")

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✓ Nós definidos


## Célula 13 — Montar e compilar o grafo

In [16]:
# Célula 13 — Montar o grafo LangGraph
grafo = StateGraph(EstadoConsulta)

# Adiciona os nós
grafo.add_node("triagem",       no_triagem)
grafo.add_node("busca_rag",     no_busca_rag)
grafo.add_node("gerar_resposta", no_gerar_resposta)
grafo.add_node("formatar_saida", no_formatar_saida)

# Define o fluxo
grafo.set_entry_point("triagem")
grafo.add_edge("triagem",        "busca_rag")
grafo.add_edge("busca_rag",      "gerar_resposta")
grafo.add_edge("gerar_resposta", "formatar_saida")
grafo.add_edge("formatar_saida", END)

# Compila
assistente = grafo.compile()

print("✓ Grafo compilado com sucesso")

✓ Grafo compilado com sucesso


## Célula 14 — Testar os 3 níveis de urgência

In [17]:
# Célula 14 — Testes com os 3 níveis de urgência
perguntas_teste = [
    # baixa
    "Quais vacinas são recomendadas para mulheres adultas?",
    # media
    "Paciente com suspeita de vaginose bacteriana, quais os critérios diagnósticos?",
    # alta
    "Paciente com hemorragia pós-parto intensa e queda de pressão, conduta imediata?"
]

for pergunta in perguntas_teste:
    print(f"\n{'─' * 60}")
    print(f"PERGUNTA: {pergunta}")
    assistente.invoke({
        "pergunta":       pergunta,
        "urgencia":       "",
        "contexto":       "",
        "resposta":       "",
        "encaminhamento": ""
    })


────────────────────────────────────────────────────────────
PERGUNTA: Quais vacinas são recomendadas para mulheres adultas?
[Triagem] Urgência classificada: baixa
[RAG] Contexto recuperado: 2472 caracteres
[LLM] Resposta gerada: 2467 caracteres

  ASSISTENTE DE SAÚDE DA MULHER  🟢 Urgência: BAIXA
**Vacinas recomendadas para mulheres adultas (20 anos ou mais)**  

| Vacina | Quando fazer | Observações importantes |
|--------|--------------|------------------------|
| **HPV (Papilomavírus Humano)** | Até 45 anos (3 doses se ainda não vacinada) | Protege contra tipos que causam câncer do colo do útero e verrugas genitais. A vacina está disponível gratuitamente no SUS. |
| **Hepatite B** | 3 doses se não vacinada | Essencial para prevenir hepatite B, que pode levar a cirrose e câncer de fígado. |
| **dT ou dTpa (tétano e difteria)** | Cada 10 anos; dTpa na gestação entre 20 e 36 semanas | A dose de dTpa na gestação protege o recém‑nascido contra tétano neonatal. |
| **Febre Amarela** | Do

## Célula 15 — README do projeto

In [ ]:
# Célula 15 — Gerar README.md do projeto
import os

readme = '# Tech Challenge Fase 3 — Assistente de Saúde da Mulher\n## Arquitetura RAG + Fine-Tuning (QLoRA) + LangGraph\n\n### Visão Geral\nAssistente médico especializado em saúde da mulher. Combina duas abordagens:\n- **RAG** (Retrieval-Augmented Generation) orquestrado via LangGraph;\n- **Fine-tuning** de um LLM open-source com dados médicos, usando QLoRA.\n\n### Stack Tecnológica\n| Componente         | Tecnologia                          |\n|--------------------|-------------------------------------|\n| LLM (RAG)          | LLaMA 3.1 via OpenRouter (free)     |\n| LLM (fine-tuning)  | google/gemma-2-2b-it + QLoRA        |\n| Fine-tuning        | PEFT (LoRA 4-bit) + TRL SFTTrainer  |\n| Orquestração       | LangGraph                           |\n| RAG / Embeddings   | LangChain + ChromaDB                |\n| Embeddings model   | all-MiniLM-L6-v2                    |\n| Anonimização       | Regex + spaCy NER (pt_core_news_sm) |\n| Ambiente           | Google Colab (GPU T4)               |\n\n### Dataset\n| Fonte                               | Registros | Idioma |\n|-------------------------------------|-----------|--------|\n| Sintético (protocolos SBG/FEBRASGO) | 35        | pt-BR  |\n| MedQuAD - NIH                       | ~281      | en     |\n| Menstrual Health Awareness Dataset  | ~516      | en     |\n| **Total**                           | **832**   |        |\n\n### Preparação dos Dados (Etapa 4)\n1. **Preprocessing** — normalização Unicode, remoção de HTML, limpeza de espaços.\n2. **Anonimização** — remoção de PII via regex (CPF, telefone, e-mail, datas,\n   prontuário, cartão SUS) e NER spaCy (nomes e locais). Gera o arquivo\n   `dataset_anonimizado.json`.\n3. **Curadoria** — deduplicação por pergunta, filtro de qualidade (respostas\n   curtas), truncamento de respostas longas. Split 85/15 treino/validação.\n\n### Pipeline de Fine-Tuning (QLoRA)\n- Modelo base `google/gemma-2-2b-it` carregado em 4-bit (NF4).\n- Adaptadores LoRA (r=16, alpha=32) treinados com TRL `SFTTrainer`.\n- 3 épocas, otimizador `paged_adamw_8bit`, lr 2e-4, ~15-20 min em GPU T4.\n- Avaliação base vs. fine-tuned com métricas ROUGE e comparação qualitativa.\n\n### Pipeline RAG\n```\nPergunta → [Triagem] → [Busca ChromaDB] → [LLM] → [Resposta formatada]\n```\n\n### Nós LangGraph\n- **triagem**: classifica urgência (alta/média/baixa) por palavras-chave\n- **busca_rag**: recupera top-3 documentos por similaridade semântica\n- **gerar_resposta**: envia contexto + pergunta ao LLM com prompt médico\n- **formatar_saida**: exibe resposta com emoji de urgência\n\nA Célula 28 reconstrói esse fluxo usando o modelo fine-tuned (encapsulado em\n`HuggingFacePipeline`) no lugar do LLM do OpenRouter.\n\n### Como executar\n1. Abra o notebook no Google Colab e selecione o runtime **T4 GPU**.\n2. Configure nos *Secrets* do Colab:\n   - `OPENROUTER_API_KEY` — chave do OpenRouter (etapa RAG);\n   - `HF_TOKEN` — token HuggingFace, com a licença do Gemma aceita.\n3. Execute as células em ordem:\n   - Células 1–16: dataset, RAG e LangGraph;\n   - Células 17–28: anonimização, curadoria, fine-tuning e avaliação.\n\n### Segurança e Validação\n- O assistente nunca prescreve medicamentos diretamente — sempre orienta a\n  validação por um profissional de saúde.\n- Cada resposta do RAG indica a fonte do conhecimento utilizado (explainability).\n- A triagem de urgência emite alerta para casos graves.\n'

os.makedirs("/content/drive/MyDrive/TechChallengeFase3_IA_Devs", exist_ok=True)

with open("/content/drive/MyDrive/TechChallengeFase3_IA_Devs/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("✓ README.md salvo no Drive")


## Célula 16 — Salvar datasets no Drive

In [20]:
# Célula 16 — Backup dos datasets processados no Drive
import shutil

base_drive = "/content/drive/MyDrive/TechChallengeFase3_IA_Devs"
os.makedirs(f"{base_drive}/data", exist_ok=True)

shutil.copy(
    "/content/data/processed/dataset_final.json",
    f"{base_drive}/data/dataset_final.json"
)
shutil.copy(
    "/content/data/processed/dataset_ptbr.json",
    f"{base_drive}/data/dataset_ptbr.json"
)

print("✓ Arquivos salvos no Drive:")
print(f"  → {base_drive}/data/dataset_final.json")
print(f"  → {base_drive}/data/dataset_ptbr.json")
print(f"  → {base_drive}/README.md")
print("\n✓ Projeto concluído!")

✓ Arquivos salvos no Drive:
  → /content/drive/MyDrive/TechChallengeFase3_IA_Devs/data/dataset_final.json
  → /content/drive/MyDrive/TechChallengeFase3_IA_Devs/data/dataset_ptbr.json
  → /content/drive/MyDrive/TechChallengeFase3_IA_Devs/README.md

✓ Projeto concluído!


# Etapa 4 — Fine-Tuning de LLM com QLoRA

Esta etapa cumpre o **Requisito 1** do Tech Challenge: fine-tuning de um modelo
LLM com dados médicos internos, incluindo **preprocessing, anonimização e
curadoria**.

**Modelo base:** `google/gemma-2-2b-it` &nbsp;(fallback sem licença: `Qwen/Qwen2.5-3B-Instruct`)
**Técnica:** QLoRA (LoRA com quantização 4-bit)

> ⚠️ **Antes de executar:** selecione um runtime com GPU em
> *Ambiente de execução → Alterar o tipo de ambiente → T4 GPU*.
> As células 1–16 (dataset + RAG + LangGraph) devem ter sido executadas antes.
> Se ocorrer erro de versão de biblioteca, reinicie o ambiente e execute a
> partir da Célula 17.

## Célula 17 — Instalação das bibliotecas de fine-tuning

In [ ]:
# ============================================================
# CELULA 17 -- Bibliotecas de fine-tuning (QLoRA)
# ============================================================
# Versoes fixadas para garantir a compatibilidade da API usada abaixo.
# Instalacao LIMPA: desinstala o que estiver pre-instalado antes,
# evitando um 'mixed install' do transformers (causa do ImportError).
!pip uninstall -y -q transformers tokenizers trl peft accelerate datasets

!pip install -q \
    "transformers==4.44.2" \
    "tokenizers==0.19.1" \
    "trl==0.9.6" \
    "peft==0.12.0" \
    "accelerate==0.34.2" \
    "datasets==2.21.0" \
    bitsandbytes \
    evaluate rouge_score \
    spacy sentencepiece protobuf matplotlib

# langchain-huggingface instalado SEM dependencias (--no-deps) para nao
# sobrescrever o transformers fixado acima.
!pip install -q --no-deps langchain-huggingface

!python -m spacy download pt_core_news_sm

import torch
assert torch.cuda.is_available(), \
    "GPU nao detectada! Selecione o runtime T4 GPU e execute novamente."
print("GPU:", torch.cuda.get_device_name(0))

print("\n[ATENCAO] REINICIE O RUNTIME agora: menu 'Ambiente de execucao' > 'Reiniciar sessao'.")
print("          Depois execute a partir da Celula 18 - NAO rode esta celula novamente.")

## Célula 18 — Autenticação no HuggingFace

Crie um token em https://huggingface.co/settings/tokens e aceite a licença do
Gemma em https://huggingface.co/google/gemma-2-2b-it. Adicione o token nos
*Secrets* do Colab com o nome `HF_TOKEN`.

In [ ]:
# CÉLULA 18 — Autenticação no HuggingFace
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HuggingFace autenticado ✓")

## Célula 19 — Preprocessing e Anonimização dos dados

Aplica limpeza textual e **anonimização de PII** (dado pessoal) — exigência
obrigatória do desafio:
- **Regex** para identificadores estruturados (CPF, telefone, e-mail, datas,
  prontuário, cartão SUS);
- **NER (spaCy)** para nomes de pessoas e locais nos registros em português.

In [ ]:
# CÉLULA 19 — Preprocessing + Anonimização (PII)
import json, re, os, unicodedata

# ── Carrega o dataset final produzido na Etapa 1 ─────────────
caminhos_possiveis = [
    "/content/data/processed/dataset_final.json",
    "/content/drive/MyDrive/TechChallengeFase3_IA_Devs/data/dataset_final.json",
    "dataset_final.json",
]
caminho_dataset = next((c for c in caminhos_possiveis if os.path.exists(c)), None)
assert caminho_dataset, "dataset_final.json nao encontrado — execute a Etapa 1."
with open(caminho_dataset, encoding="utf-8") as f:
    dataset = json.load(f)
print(f"Dataset carregado: {len(dataset)} registros  ({caminho_dataset})")

# ── Preprocessing: limpeza textual ───────────────────────────
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = unicodedata.normalize("NFKC", texto)
    texto = re.sub(r"<[^>]+>", " ", texto)      # remove tags HTML
    texto = re.sub(r"[ \t]+", " ", texto)       # colapsa espacos
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

# ── Anonimização: regex para identificadores estruturados ────
PADROES = {
    "[CPF]":        re.compile(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b"),
    "[TELEFONE]":   re.compile(r"\b(?:\+?55\s?)?\(?\d{2}\)?\s?9?\d{4}-?\d{4}\b"),
    "[EMAIL]":      re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "[DATA]":       re.compile(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b"),
    "[PRONTUARIO]": re.compile(r"\b(?:prontu[áa]rio|matr[íi]cula|registro)\s*n?[ºn.:]*\s*\d+\b", re.IGNORECASE),
    "[CNS]":        re.compile(r"\b\d{3}\s\d{4}\s\d{4}\s\d{4}\b"),
}

import spacy
nlp = spacy.load("pt_core_news_sm")

contador = {tag: 0 for tag in PADROES}
contador["[NOME]"] = 0
contador["[LOCAL]"] = 0

def anonimizar(texto, usar_ner=True):
    # 1) Identificadores estruturados via regex
    for tag, padrao in PADROES.items():
        texto, n = padrao.subn(tag, texto)
        contador[tag] += n
    # 2) Nomes e locais via NER (apenas textos em pt-BR)
    if usar_ner:
        doc = nlp(texto)
        repl = []
        for ent in doc.ents:
            if ent.label_ == "PER":
                repl.append((ent.start_char, ent.end_char, "[NOME]"))
            elif ent.label_ == "LOC":
                repl.append((ent.start_char, ent.end_char, "[LOCAL]"))
        for ini, fim, tag in sorted(repl, reverse=True):
            texto = texto[:ini] + tag + texto[fim:]
            contador[tag] += 1
    return texto

for item in dataset:
    eh_ptbr = item.get("idioma") == "pt-BR"
    item["pergunta"] = anonimizar(limpar_texto(item["pergunta"]), usar_ner=eh_ptbr)
    item["resposta"] = anonimizar(limpar_texto(item["resposta"]), usar_ner=eh_ptbr)

print("\nEntidades anonimizadas (numero de substituicoes):")
for tag, n in contador.items():
    print(f"  {tag:13s} {n}")

os.makedirs("/content/data/processed", exist_ok=True)
caminho_anon = "/content/data/processed/dataset_anonimizado.json"
with open(caminho_anon, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)
print(f"\n✓ Dataset anonimizado salvo: {caminho_anon}")

## Célula 20 — Curadoria e filtragem de qualidade

In [ ]:
# CÉLULA 20 — Curadoria: deduplicacao e filtros de qualidade
def normalizar(texto):
    return re.sub(r"\s+", " ", texto.lower().strip())

def truncar_sentenca(texto, limite=1500):
    if len(texto) <= limite:
        return texto
    corte = texto[:limite]
    fim = max(corte.rfind(". "), corte.rfind("! "), corte.rfind("? "))
    return corte[:fim + 1].strip() if fim > limite * 0.5 else corte.strip()

def filtrar_e_curar(dados):
    vistos, curado = set(), []
    stats = {"resposta_curta": 0, "duplicado": 0, "truncado": 0}
    for item in dados:
        p, r = item["pergunta"].strip(), item["resposta"].strip()
        if len(p) < 10 or len(r) < 80:
            stats["resposta_curta"] += 1
            continue
        chave = normalizar(p)
        if chave in vistos:
            stats["duplicado"] += 1
            continue
        vistos.add(chave)
        if len(r) > 1500:
            r = truncar_sentenca(r)
            stats["truncado"] += 1
        curado.append({**item, "pergunta": p, "resposta": r})
    return curado, stats

dataset_curado, stats = filtrar_e_curar(dataset)
print(f"Registros antes da curadoria:         {len(dataset)}")
print(f"  - removidos (resposta < 80 chars):   {stats['resposta_curta']}")
print(f"  - removidos (pergunta duplicada):    {stats['duplicado']}")
print(f"  - respostas truncadas (> 1500):      {stats['truncado']}")
print(f"Registros apos a curadoria:           {len(dataset_curado)}")

import pandas as pd
df_curado = pd.DataFrame(dataset_curado)
print("\nDistribuicao por idioma:")
print(df_curado["idioma"].value_counts().to_string())
print("\nDistribuicao por categoria:")
print(df_curado["categoria"].value_counts().to_string())

## Célula 21 — Conversão para formato de instrução (chat)

In [ ]:
# CÉLULA 21 — Formato de instrucao (mensagens) + split treino/validacao
from sklearn.model_selection import train_test_split
from datasets import Dataset

SYSTEM_PROMPT = (
    "Voce e um assistente medico especializado em saude da mulher. "
    "Responda sempre em portugues brasileiro, de forma clara, empatica e "
    "baseada em evidencias. Nunca prescreva medicamentos diretamente: "
    "sempre oriente a validacao por um profissional de saude."
)

def para_mensagens(item):
    # O system prompt e embutido no turno do usuario porque o chat template
    # do Gemma nao aceita o papel 'system' separadamente.
    contexto = f"[Categoria: {item['categoria']} | Urgencia: {item['urgencia']}]"
    conteudo_usuario = f"{SYSTEM_PROMPT}\n\n{contexto}\n{item['pergunta']}"
    return {"messages": [
        {"role": "user",      "content": conteudo_usuario},
        {"role": "assistant", "content": item["resposta"]},
    ]}

registros = [para_mensagens(x) for x in dataset_curado]
treino, validacao = train_test_split(registros, test_size=0.15, random_state=42)

ds_treino    = Dataset.from_list(treino)
ds_validacao = Dataset.from_list(validacao)
print(f"Exemplos de treino:    {len(ds_treino)}")
print(f"Exemplos de validacao: {len(ds_validacao)}")
print("\nExemplo de entrada formatada:")
print(ds_treino[0]["messages"][0]["content"][:300], "...")

## Célula 22 — Carregar o modelo base com quantização 4-bit

In [ ]:
# Compatibilidade: Colab 2025 usa transformers>=4.50 que removeu suporte a Flax.
# gemma/__init__.py ainda importa is_flax_available -> injetamos de volta.
import transformers
import transformers.utils as _tu
print("transformers carregado:", transformers.__version__, "--", transformers.__file__)
if not hasattr(_tu, "is_flax_available"):
    _tu.is_flax_available = lambda: False
    print("Patch aplicado: is_flax_available = lambda: False")
else:
    print("Sem patch necessario: is_flax_available ja existe")
del _tu

In [ ]:
# CÉLULA 22 — Modelo base em 4-bit (QLoRA)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELO_BASE = "google/gemma-2-2b-it"
# Fallback sem restricao de licenca: "Qwen/Qwen2.5-3B-Instruct"

# T4 (Turing) nao tem bf16 nativo -> usa fp16. Ampere+ (A100/L4) -> bf16.
USA_BF16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if USA_BF16 else torch.float16
print(f"Compute dtype: {'bfloat16' if USA_BF16 else 'float16'}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=DTYPE,
    attn_implementation="eager",   # recomendado para a familia Gemma-2
)
modelo.config.use_cache = False
print(f"✓ Modelo {MODELO_BASE} carregado em 4-bit")

## Célula 23 — Configurar os adaptadores LoRA

In [ ]:
# CÉLULA 23 — Adaptadores LoRA
from peft import (LoraConfig, prepare_model_for_kbit_training,
                  get_peft_model, TaskType)

modelo = prepare_model_for_kbit_training(modelo)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
modelo = get_peft_model(modelo, lora_config)
modelo.print_trainable_parameters()

## Célula 24 — Treinamento com SFTTrainer

In [ ]:
# CÉLULA 24 — Treinamento supervisionado (SFT) com QLoRA
import os
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# Reduz fragmentação de memória CUDA
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

# Aplica o chat template -> coluna de texto unica para o SFTTrainer
def aplicar_template(exemplo):
    return {"text": tokenizer.apply_chat_template(
        exemplo["messages"], tokenize=False)}

ds_treino_fmt    = ds_treino.map(aplicar_template, remove_columns=["messages"])
ds_validacao_fmt = ds_validacao.map(aplicar_template, remove_columns=["messages"])

OUTPUT_DIR = "/content/drive/MyDrive/TechChallengeFase3_IA_Devs/checkpoints"

# batch efetivo = per_device_train_batch_size * gradient_accumulation_steps = 2 * 8 = 16
# (igual ao original 4 * 4 = 16, mas com muito menos pico de VRAM por step)
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,      # era 4 -> reduz pico de VRAM por step
    gradient_accumulation_steps=8,      # era 4 -> compensa, batch efetivo=16
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=not USA_BF16,
    bf16=USA_BF16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo,
    tokenizer=tokenizer,
    args=args,
    train_dataset=ds_treino_fmt,
    eval_dataset=ds_validacao_fmt,
    dataset_text_field="text",
    max_seq_length=512,     # era 1024 -> maior economia de VRAM (logits proporcional a seq_len x vocab_size)
    packing=False,
)

trainer.train()
print("✓ Treinamento concluido")

## Célula 25 — Salvar o adaptador LoRA no Google Drive

In [ ]:
# CÉLULA 25 — Salvar o adaptador treinado
CAMINHO_MODELO = "/content/drive/MyDrive/TechChallengeFase3_IA_Devs/modelo_finetuned"

trainer.model.save_pretrained(CAMINHO_MODELO)
tokenizer.save_pretrained(CAMINHO_MODELO)

print("✓ Adaptador LoRA + tokenizer salvos em:")
print(f"  {CAMINHO_MODELO}")

## Célula 25b — Publicar o adaptador LoRA no Hugging Face Hub

In [ ]:
# CÉLULA 25b — Push do adaptador LoRA para o Hugging Face Hub
# Pré-requisito: HF_TOKEN já configurado na Célula 18.
# Altere REPO_ID para <seu-usuario-hf>/<nome-do-repositorio>

REPO_ID = "SEU_USUARIO_HF/saude-da-mulher-gemma2-lora"  # <-- edite aqui

# Publica o adaptador LoRA (arquivos adapter_config.json + adapter_model.safetensors)
trainer.model.push_to_hub(REPO_ID, private=True)

# Publica o tokenizer junto
tokenizer.push_to_hub(REPO_ID, private=True)

print(f"Adaptador LoRA publicado em: https://huggingface.co/{REPO_ID}")
print("Repositorio criado como PRIVADO. Para tornar publico acesse o HF Hub e altere a visibilidade.")


## Célula 26 — Avaliação: modelo base vs. fine-tuned

In [ ]:
# CÉLULA 26 — Avaliacao qualitativa e quantitativa
import contextlib, evaluate, pandas as pd

modelo.config.use_cache = True
modelo.eval()

def gerar(mensagens, com_adaptador=True, max_new=220):
    prompt = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    cm = contextlib.nullcontext() if com_adaptador else modelo.disable_adapter()
    with cm, torch.no_grad():
        saida = modelo.generate(
            **inputs, max_new_tokens=max_new, do_sample=False,
            pad_token_id=tokenizer.pad_token_id)
    novos = saida[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(novos, skip_special_tokens=True).strip()

def msg_usuario(pergunta):
    return [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{pergunta}"}]

# ── Avaliacao qualitativa: 5 perguntas (inclui 1 fora de dominio) ──
perguntas_qual = [
    "Quais sao os sintomas mais comuns da sindrome do ovario policistico?",
    "Paciente com hemorragia pos-parto intensa e queda de pressao. Conduta imediata?",
    "Quais metodos contraceptivos sao seguros para uma paciente hipertensa?",
    "What are the common symptoms of menstrual cramps?",
    "Qual a melhor estrategia para investir na bolsa de valores?",
]
for p in perguntas_qual:
    print("=" * 72)
    print("PERGUNTA:", p)
    print("\n[ BASE       ]", gerar(msg_usuario(p), com_adaptador=False))
    print("\n[ FINE-TUNED ]", gerar(msg_usuario(p), com_adaptador=True))
    print()

# ── Avaliacao quantitativa: ROUGE no conjunto de validacao ─────────
rouge = evaluate.load("rouge")
amostra = validacao[:15]
refs, pred_base, pred_ft = [], [], []
for ex in amostra:
    msg = [ex["messages"][0]]
    refs.append(ex["messages"][1]["content"])
    pred_base.append(gerar(msg, com_adaptador=False, max_new=200))
    pred_ft.append(gerar(msg, com_adaptador=True, max_new=200))

r_base = rouge.compute(predictions=pred_base, references=refs)
r_ft   = rouge.compute(predictions=pred_ft,   references=refs)
tabela = pd.DataFrame({"Modelo Base": r_base, "Fine-Tuned": r_ft}).round(4)
print("=" * 72)
print("METRICAS ROUGE (conjunto de validacao, n=15)")
print(tabela.to_string())

## Célula 27 — Visualização da curva de loss

In [ ]:
# CÉLULA 27 — Curva de loss (treino vs. validacao)
import matplotlib.pyplot as plt

hist = trainer.state.log_history
treino_loss = [(h["step"], h["loss"])      for h in hist if "loss" in h]
val_loss    = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

plt.figure(figsize=(9, 5))
if treino_loss:
    plt.plot(*zip(*treino_loss), label="Loss de treino")
if val_loss:
    plt.plot(*zip(*val_loss), label="Loss de validacao", marker="o")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Curva de Loss — Fine-Tuning QLoRA (Gemma-2-2B)")
plt.legend()
plt.grid(alpha=0.3)

caminho_png = "/content/drive/MyDrive/TechChallengeFase3_IA_Devs/curva_loss.png"
plt.savefig(caminho_png, dpi=120, bbox_inches="tight")
plt.show()
print(f"✓ Grafico salvo: {caminho_png}")

## Célula 28 — Integração do modelo fine-tuned ao LangGraph

Reconstrói o fluxo LangGraph da Etapa 3 substituindo o LLM do OpenRouter pelo
modelo treinado localmente, encapsulado em `HuggingFacePipeline` do LangChain.

In [ ]:
# CÉLULA 28 — Modelo fine-tuned dentro do fluxo LangGraph
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

gerador = pipeline(
    "text-generation",
    model=modelo,
    tokenizer=tokenizer,
    max_new_tokens=400,
    do_sample=False,
    return_full_text=False,
)
llm_finetuned = HuggingFacePipeline(pipeline=gerador)

# Novo no de geracao que usa o modelo local (substitui o ChatOpenAI)
def no_gerar_resposta_local(estado):
    alerta = ""
    if estado["urgencia"] == "alta":
        alerta = "\n⚠️ ATENCAO: Caso de ALTA URGENCIA — considere atendimento imediato.\n"
    conteudo = (
        f"{SYSTEM_PROMPT}{alerta}\n\n"
        f"Contexto medico relevante:\n{estado['contexto']}\n\n"
        f"Pergunta:\n{estado['pergunta']}"
    )
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": conteudo}],
        tokenize=False, add_generation_prompt=True)
    resposta = llm_finetuned.invoke(prompt)
    print(f"[LLM fine-tuned] Resposta gerada: {len(resposta)} caracteres")
    return {**estado, "resposta": resposta.strip()}

# Reconstroi o grafo reaproveitando os nos definidos na Etapa 3
grafo_local = StateGraph(EstadoConsulta)
grafo_local.add_node("triagem",        no_triagem)
grafo_local.add_node("busca_rag",      no_busca_rag)
grafo_local.add_node("gerar_resposta", no_gerar_resposta_local)
grafo_local.add_node("formatar_saida", no_formatar_saida)
grafo_local.set_entry_point("triagem")
grafo_local.add_edge("triagem",        "busca_rag")
grafo_local.add_edge("busca_rag",      "gerar_resposta")
grafo_local.add_edge("gerar_resposta", "formatar_saida")
grafo_local.add_edge("formatar_saida", END)
assistente_local = grafo_local.compile()
print("✓ Grafo LangGraph com modelo fine-tuned compilado\n")

# Testa os 3 niveis de urgencia usando o modelo fine-tuned
perguntas_teste_ft = [
    "Quais vacinas sao recomendadas para mulheres adultas?",
    "Paciente com suspeita de vaginose bacteriana, quais os criterios diagnosticos?",
    "Paciente com hemorragia pos-parto intensa e queda de pressao, conduta imediata?",
]
for pergunta in perguntas_teste_ft:
    print(f"\n{'-' * 60}")
    print(f"PERGUNTA: {pergunta}")
    assistente_local.invoke({
        "pergunta": pergunta, "urgencia": "", "contexto": "",
        "resposta": "", "encaminhamento": "",
    })